# Hexagonal vs. Square Kernels

Port of HexagDLy's own `hexagdly_hex_vs_square.ipynb`: a benchmark comparison between hexagonal kernels (`keras_hexagdly`) and standard square kernels (plain Keras `Conv2D`).

**Not ported**: upstream's resampling-artifact section, which interpolates a hex grid onto a finer square grid with `scipy.interpolate.griddata` to show sampling artifacts. That step is framework-agnostic pre-processing unrelated to either HexagDLy or `keras_hexagdly` -- see upstream's notebook directly if you need it.

## Parameter count and raw op speed

A hexagonal kernel that covers a given neighbourhood radius needs fewer weights than a square kernel covering a comparable area (`1 + 3n(n+1)` cells for hex radius `n`, vs. `(2n+1)^2` for an n-ring square kernel -- the square kernel always includes extra corner pixels outside the hex neighbourhood).

In [1]:
import time
import numpy as np
import keras
import keras_hexagdly as hgly

H, W, Cin, Cout = 100, 100, 3, 1
hexconv = hgly.Conv2d(Cin, Cout, kernel_size=1, stride=1)
squareconv = keras.layers.Conv2D(Cout, 3, padding='same')

x = np.random.randn(10, H, W, Cin).astype('float32')
_ = hexconv(x); _ = squareconv(x)  # build
print('hex kernel (radius 1) parameters:   ', hexconv.count_params())
print('square kernel (3x3) parameters:     ', squareconv.count_params())

hex kernel (radius 1) parameters:    22
square kernel (3x3) parameters:      28


This does **not** mean the hexagonal convolution is faster in wall-clock time: `keras_hexagdly`, like upstream HexagDLy, decomposes a hex kernel into several sub-convolutions on shifted/padded slices of the input, which costs more op-dispatch overhead than the single, heavily-optimized square `Conv2D`. Both implementations are prototyping tools that favour flexibility over raw performance (see the disclaimer in the README) -- this is visible directly:

In [2]:
n_reps = 50

t0 = time.time()
for _ in range(n_reps):
    hexconv(x)
t1 = time.time()
for _ in range(n_reps):
    squareconv(x)
t2 = time.time()

print(f'hex conv:    {(t1 - t0) / n_reps * 1000:.2f} ms/call')
print(f'square conv: {(t2 - t1) / n_reps * 1000:.2f} ms/call')

hex conv:    4.77 ms/call
square conv: 0.96 ms/call


## A small CNN: hex kernels vs. square kernels

To see whether respecting the hexagonal symmetry matters beyond a single operation, two small CNNs are trained on the same toy classification task from `keras_hexagdly_cnn_example.ipynb`: one using `keras_hexagdly` layers, the other using plain square `Conv2D`/`MaxPooling2D` with a 5x5 kernel (chosen to cover a comparable neighbourhood to the hex kernel's 2 rings, at the cost of including the extra corner pixels noted above).

In [3]:
from toy_data import ToyDataset

shape_list = ['snowflake_2', 'snowflake_3', 'snowflake_4', 'double_hex']
H, W = 20, 20

train_data = ToyDataset(shape_list, 128, H, W).create(seed=0)
val_data = ToyDataset(shape_list, 32, H, W).create(seed=1)
x_train, y_train = train_data.to_arrays()
x_val, y_val = val_data.to_arrays()


def hex_model():
    return keras.Sequential([
        keras.layers.Input(shape=(H, W, 1)),
        hgly.Conv2d(out_channels=16, kernel_size=2, stride=1),
        keras.layers.ReLU(),
        hgly.MaxPool2d(kernel_size=1, stride=2),
        hgly.Conv2d(out_channels=32, kernel_size=2, stride=1),
        keras.layers.ReLU(),
        hgly.MaxPool2d(kernel_size=1, stride=2),
        keras.layers.Flatten(),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dense(len(shape_list)),
    ])


def square_model():
    return keras.Sequential([
        keras.layers.Input(shape=(H, W, 1)),
        keras.layers.Conv2D(16, 5, padding='same'),
        keras.layers.ReLU(),
        keras.layers.MaxPooling2D(2, strides=2, padding='same'),
        keras.layers.Conv2D(32, 5, padding='same'),
        keras.layers.ReLU(),
        keras.layers.MaxPooling2D(2, strides=2, padding='same'),
        keras.layers.Flatten(),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dense(len(shape_list)),
    ])

In [4]:
results = {}
for name, build in [('hex', hex_model), ('square', square_model)]:
    model = build()
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy'],
    )
    t0 = time.time()
    history = model.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=25, verbose=0)
    t1 = time.time()
    results[name] = dict(
        params=model.count_params(),
        train_time_s=t1 - t0,
        val_accuracy=history.history['val_accuracy'][-1],
    )

for name, r in results.items():
    print(f"{name:>7s}: {r['params']:6d} params, "
          f"{r['train_time_s']:5.2f}s training, "
          f"{100 * r['val_accuracy']:5.1f}% val accuracy")

    hex:  35844 params,  5.96s training, 100.0% val accuracy
 square:  39012 params,  2.36s training,  99.2% val accuracy


On this simple, well-separated toy task both models tend to reach similar accuracy -- the symmetry-breaking effect of a square kernel (clearly visible at the single-operation level in `keras_hexagdly_2d_example.ipynb`) doesn't necessarily show up as an accuracy gap once a deep-enough CNN is trained on a small, easy classification task. The practical trade-off this notebook actually demonstrates is: hex kernels need fewer parameters per receptive-field radius, at the cost of more op-dispatch overhead per call (sub-kernel decomposition vs. a single fused square conv).